# Session 3: Generative Models, Likelihood and Prediction
### Student Laboratory Workbook
*Course: Bayesian Analysis of Empirical Data (2026)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/03_generative_models_likelihood_prediction.ipynb)

---

## 1. Before Class
* **Goal**: Translate a substantive data-generating story into forward simulation, evaluate likelihood compatibility over candidate parameters, and distinguish parameter uncertainty (epistemic) from observable predictive variation (aleatory).
* **Expected Runtime**: ~90 minutes (Classwork: 45 min, Independent Practice: 30 min, Reflection: 15 min).
* **Make Your Copy**: Click **File $\to$ Save a copy in Drive** to preserve your own edits and code experiments.


## 2. Environment Check & Setup
Run this cell first to initialize imports, set random seeds, and configure inline Plotly rendering for Google Colab.


In [ ]:
# ==============================================================================
# 🚀 1. Setup Cell: Environment & Reproducibility Check
# ==============================================================================
import sys
import os
import numpy as np
import pandas as pd
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("⚡ Running in Google Colab environment.")
    import plotly.io as pio
    pio.renderers.default = "colab"
else:
    print("💻 Running in local environment.")

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
print(f"✅ Environment initialized. NumPy random seed set to {RANDOM_SEED}.")


## 3. Observable Learning Targets
By completing this workbook, you will be able to:
1. **Simulate forward**: Write Python code using `scipy.stats` and `numpy.random` that generates synthetic datasets from explicit generative parameters.
2. **Evaluate likelihoods**: Plot $\mathcal{L}(	heta \mid y)$ by holding the observed empirical evidence fixed and sweeping candidate parameters.
3. **Contrast the Bayesian Quartet**: Formally distinguish Prior $p(	heta)$, Prior Predictive $p(y_{	ext{sim}})$, Posterior $p(	heta \mid y)$, and Posterior Predictive $p(	ilde{y} \mid y)$.
4. **Decouple expectation from realization**: Distinguish predictions of the expected average ($\mathbb{E}[	ilde{y} \mid 	heta]$) from individual future realizations ($	ilde{y}$).
5. **Diagnose model breakdown**: Identify when count data exhibit overdispersion ($	ext{Var} > 	ext{Mean}$) and motivate mixture models (Negative Binomial).


## 4. Classwork 0: Predict Before Running
> ✍ **WRITE (Prediction)**: Suppose a poll of $N = 20$ voters yields $k = 13$ supporters of a policy.
> 1. Which value of the underlying support rate $	heta \in [0, 1]$ will maximize the likelihood $\mathcal{L}(	heta \mid k=13)$?
> 2. Will the likelihood function over $	heta$ integrate to $1.0$? Why or why not?


---
## 5. Classwork 1: Forward Generative Simulation (▶ RUN TOGETHER)
We begin with a known "ground truth" parameter $	heta_0 = 0.65$ (e.g. true public support) and simulate $S = 1,000$ independent survey samples of size $N = 20$.


In [ ]:
N_trials = 20
theta_true = 0.65
S_sims = 1000

# Forward simulation: drawing from Binomial(N=20, theta=0.65)
simulated_k = rng.binomial(n=N_trials, p=theta_true, size=S_sims)

print(f"Simulated {S_sims} survey replications.")
print(f"Theoretical Mean:   {N_trials * theta_true:.2f}")
print(f"Empirical Mean:     {simulated_k.mean():.2f}")
print(f"Theoretical StdDev: {np.sqrt(N_trials * theta_true * (1 - theta_true)):.2f}")
print(f"Empirical StdDev:   {simulated_k.std():.2f}")

# Histogram of simulated observations
counts, bins = np.histogram(simulated_k, bins=np.arange(-0.5, N_trials + 1.5, 1))
fig_sim = go.Figure(go.Bar(
    x=np.arange(0, N_trials + 1),
    y=counts / S_sims,
    marker_color='#2b6cb0',
    hovertemplate='<b>k = %{x}</b><br>Simulated Relative Frequency: %{y:.3f}<extra></extra>'
))
fig_sim.update_layout(
    title=f'Forward Simulation (Generative World): Binomial(N={N_trials}, θ={theta_true})',
    xaxis_title='Simulated Successes k (out of 20)',
    yaxis_title='Relative Frequency across 1,000 Simulations',
    template='plotly_white',
    height=400
)
fig_sim.show()


### 🛑 STOP 1: Checkpoint
* Check your plot above: Are outcomes concentrated around $k = 13$?
* Notice that even with a fixed, known model $	heta = 0.65$, individual sample outcomes vary anywhere from $k = 8$ to $k = 18$. This is **pure aleatory variability** (sampling noise).


---
## 6. Classwork 2: Inverse Reasoning — Evaluating the Likelihood Function (🧪 CHANGE ONE THING)
Now we switch to the inferential perspective. We **fix our observed sample** at $k_{	ext{obs}} = 13$ out of $N = 20$, and evaluate the compatibility of this fixed evidence across all candidate models $	heta \in [0, 1]$.


In [ ]:
k_obs = 13
N_obs = 20

theta_grid = np.linspace(0.001, 0.999, 300)

# Likelihood function: L(theta | k=13, N=20) = binom.pmf(13, 20, theta)
likelihood = stats.binom.pmf(k_obs, N_obs, theta_grid)

# Find Maximum Likelihood Estimate (MLE)
mle_theta = theta_grid[np.argmax(likelihood)]

fig_lik = go.Figure()
fig_lik.add_trace(go.Scatter(
    x=theta_grid, y=likelihood,
    mode='lines',
    line=dict(color='#d97706', width=2.5),
    fill='tozeroy',
    fillcolor='rgba(217, 119, 6, 0.12)',
    name=f'Likelihood L(θ | k={k_obs})'
))

# Highlight H1 (0.65) vs H2 (0.50)
l_65 = stats.binom.pmf(k_obs, N_obs, 0.65)
l_50 = stats.binom.pmf(k_obs, N_obs, 0.50)
lr = l_65 / l_50

fig_lik.add_trace(go.Scatter(
    x=[0.50, 0.65], y=[l_50, l_65],
    mode='markers+text',
    text=[f'H2: θ=0.50<br>(L={l_50:.4f})', f'H1: θ=0.65<br>(L={l_65:.4f})'],
    textposition=['bottom left', 'top right'],
    marker=dict(size=10, color=['#c53030', '#276749']),
    name='Hypothesis Comparison'
))

fig_lik.update_layout(
    title=f'Continuous Likelihood Function L(θ | k={k_obs}, N={N_obs}) [Likelihood Ratio H1/H2 = {lr:.2f}]',
    xaxis_title='Candidate Parameter θ (Public Support Rate)',
    yaxis_title='Likelihood Value L(θ)',
    template='plotly_white',
    height=420
)
fig_lik.show()

print(f"Maximum Likelihood Estimate (MLE): θ_hat = {mle_theta:.3f} (Sample proportion = {k_obs}/{N_obs} = {k_obs/N_obs:.3f})")
print(f"Likelihood at H1 (θ=0.65): {l_65:.4f}")
print(f"Likelihood at H2 (θ=0.50): {l_50:.4f}")
print(f"Likelihood Ratio (LR):     {lr:.3f} (The data are {lr:.2f}x more compatible with H1 than H2)")


### 🛑 STOP 2: Checkpoint & Numerical Integration
Let us verify numerically whether the likelihood integrates to 1.0!


In [ ]:
from scipy.integrate import trapezoid

area_under_likelihood = trapezoid(likelihood, theta_grid)
theoretical_area = 1.0 / (N_obs + 1)

print(f"Trapezoidal Area under Likelihood curve: {area_under_likelihood:.4f}")
print(f"Theoretical Area 1/(N+1) = 1/21:          {theoretical_area:.4f}")
print(f"✅ Check: Is the likelihood a probability density? {'YES' if np.isclose(area_under_likelihood, 1.0) else 'NO (Area ≠ 1.0)'}!")


---
## 7. Classwork 3: Predictions — Distinguishing the Bayesian Quartet (▶ RUN TOGETHER)
Now we apply Bayes' rule with a prior $	heta \sim \operatorname{Beta}(2, 2)$ to generate both **parameter uncertainty** and **predictive replications**.


In [ ]:
# 1. Prior: Beta(2, 2)
prior_a, prior_b = 2, 2
theta_prior_samples = rng.beta(prior_a, prior_b, size=2000)

# Prior predictive: draw fake data using prior parameter draws
y_prior_pred = rng.binomial(n=N_obs, p=theta_prior_samples)

# 2. Posterior: Beta(2 + 13, 2 + 7) = Beta(15, 9)
post_a = prior_a + k_obs
post_b = prior_b + (N_obs - k_obs)
theta_post_samples = rng.beta(post_a, post_b, size=2000)

# Posterior predictive: draw future data using posterior parameter draws
y_post_pred = rng.binomial(n=N_obs, p=theta_post_samples)

# Compare Prior Predictive vs Posterior Predictive
fig_quartet = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        '<b>Prior Predictive Distribution</b><br><span style="font-size:11px;color:#64748b">Predictions before seeing data (Diffused)</span>',
        '<b>Posterior Predictive Distribution</b><br><span style="font-size:11px;color:#64748b">Predictions after conditioning on k=13 (Calibrated)</span>'
    ]
)

fig_quartet.add_trace(go.Histogram(x=y_prior_pred, histnorm='probability', marker_color='#94a3b8', name='Prior Pred'), row=1, col=1)
fig_quartet.add_trace(go.Histogram(x=y_post_pred, histnorm='probability', marker_color='#2563eb', name='Posterior Pred'), row=1, col=2)

# Mark the observed data k=13
fig_quartet.add_vline(x=k_obs, line_dash='dash', line_color='#dc2626', annotation_text='Observed k=13', row=1, col=2)

fig_quartet.update_layout(template='plotly_white', height=430, showlegend=False)
fig_quartet.update_xaxes(title_text='Simulated k (out of 20)', range=[-0.5, 20.5])
fig_quartet.update_yaxes(title_text='Probability')
fig_quartet.show()


---
## 8. Exit Record
> ✍ **WRITE (Summary Reflection)**:
> 1. **Estimand**: What is the theoretical parameter in this exercise?
> 2. **Result**: What is the posterior mean and 90% credible interval for $	heta$?
> 3. **Epistemic vs. Aleatory Distinction**: Why does the posterior predictive distribution for future survey respondents have greater variance than the posterior distribution of $	heta$?


---
## 9. 🏠 Optional Homework: Diagnosing Poisson Overdispersion
In a Poisson generative process, $\mathbb{E}[Y] = \operatorname{Var}(Y) = \lambda$. But empirical cognitive and social counts frequently violate this assumption.

Run the simulation below to compare a standard Poisson model against a Negative Binomial mixture where participants have heterogeneous latent rates $\lambda_i \sim \operatorname{Gamma}(lpha, eta)$:


In [ ]:
# Sample size of 1,000 count observations
n_counts = 1000
mean_rate = 4.0

# 1. Equi-dispersed Poisson: lambda = 4.0
y_poisson = rng.poisson(lam=mean_rate, size=n_counts)

# 2. Overdispersed Negative Binomial (Gamma-Poisson mixture)
# Gamma shape alpha = 2, scale = mean_rate / alpha
alpha_shape = 2.0
lambda_individual = rng.gamma(shape=alpha_shape, scale=mean_rate / alpha_shape, size=n_counts)
y_negbin = rng.poisson(lam=lambda_individual)

print(f"Poisson:           Mean = {y_poisson.mean():.2f}, Variance = {y_poisson.var():.2f} (Ratio Var/Mean = {y_poisson.var()/y_poisson.mean():.2f})")
print(f"Negative Binomial: Mean = {y_negbin.mean():.2f}, Variance = {y_negbin.var():.2f} (Ratio Var/Mean = {y_negbin.var()/y_negbin.mean():.2f})")

fig_disp = make_subplots(rows=1, cols=2, subplot_titles=['Standard Poisson (Var = Mean = 4.0)', 'Overdispersed Negative Binomial (Var ≈ 12.0)'])
fig_disp.add_trace(go.Histogram(x=y_poisson, marker_color='#38a169', name='Poisson'), row=1, col=1)
fig_disp.add_trace(go.Histogram(x=y_negbin, marker_color='#e53e3e', name='NegBinomial'), row=1, col=2)
fig_disp.update_layout(template='plotly_white', height=400, showlegend=False)
fig_disp.update_xaxes(title_text='Count of Events')
fig_disp.show()


---
## 10. Reproducibility Footer
* Python environment: Python 3.12
* Key dependencies: `numpy`, `scipy`, `plotly`, `pandas`
* Execution environment: Self-healing Google Colab notebook
